# NS10 Tutorial D - The Copy Model

**Lecture:** NS10 - Preferential Attachment Extensions

**Reading:** Kumar et al. (2000)

## Learning goals
- Implement the lecture copy rule:
  1. choose a target node uniformly,
  2. with probability $p$, attach to the target,
  3. with probability $1-p$, attach to one of its neighbors.
- Understand how **imitation** can generate broad degree heterogeneity.
- Compare the model's tunable tail to the theoretical relation
  $$
  \gamma = \frac{2-p}{1-p}.
  $$
- Distinguish copying from explicit triadic closure.


In [ ]:
from netsci_utils import *
import pandas as pd

set_seeds()
%matplotlib inline


def copy_model(n, p, seed=RANDOM_SEED, return_log=False):
    rng = np.random.default_rng(seed)
    G = nx.Graph()
    G.add_nodes_from([0, 1])
    G.add_edge(0, 1)
    log_rows = []

    for new_node in range(2, n):
        target = int(rng.choice(list(G.nodes())))
        neighbors = list(G.neighbors(target))
        attach_to = target
        copied = False
        if neighbors and rng.random() > p:
            attach_to = int(rng.choice(neighbors))
            copied = True
        G.add_node(new_node)
        G.add_edge(new_node, attach_to)
        if return_log:
            log_rows.append({
                'new node': new_node,
                'target': target,
                'attached to': attach_to,
                'copied neighbor': copied,
            })

    if return_log:
        return G, pd.DataFrame(log_rows)
    return G


def generalized_copy_model(n, m, p, seed=RANDOM_SEED):
    rng = np.random.default_rng(seed)
    G = nx.complete_graph(max(m + 1, 3))
    for new_node in range(G.number_of_nodes(), n):
        existing = list(G.nodes())
        targets = set()
        while len(targets) < m:
            target = int(rng.choice(existing))
            neighbors = list(set(G.neighbors(target)) - targets)
            attach_to = target
            if neighbors and rng.random() > p:
                attach_to = int(rng.choice(neighbors))
            if attach_to in targets:
                remaining = list(set(existing) - targets)
                if not remaining:
                    break
                attach_to = int(rng.choice(remaining))
            targets.add(attach_to)
        G.add_node(new_node)
        for target in targets:
            G.add_edge(new_node, target)
    return G


def random_walk_attachment_model(n, m, p, seed=RANDOM_SEED):
    rng = np.random.default_rng(seed)
    G = nx.complete_graph(max(m + 1, 3))
    for new_node in range(G.number_of_nodes(), n):
        existing = list(G.nodes())
        first_target = int(rng.choice(existing))
        targets = {first_target}
        while len(targets) < m:
            if rng.random() < p and len(list(G.neighbors(first_target))) > 0:
                candidates = list(set(G.neighbors(first_target)) - targets)
                if candidates:
                    targets.add(int(rng.choice(candidates)))
                    continue
            fallback = list(set(existing) - targets)
            if not fallback:
                break
            targets.add(int(rng.choice(fallback)))
        G.add_node(new_node)
        for target in targets:
            G.add_edge(new_node, target)
    return G


def gamma_from_ccdf_slope(sequence, kmin=10):
    slope = naive_ccdf_slope(sequence, kmin=kmin)
    if np.isnan(slope):
        return np.nan
    return 1 - slope

---
## 1. Copying is an imitation mechanism

The new node does **not** need to know global degree. It only needs a target node and one of that target's neighbours.

This is why the model is natural for the Web, citation reuse, and duplication processes.


In [ ]:
copy_small, copy_log = copy_model(n=16, p=0.4, seed=RANDOM_SEED, return_log=True)
copy_small_pos = nx.spring_layout(copy_small, seed=RANDOM_SEED)

plot_graph(
    copy_small,
    pos=copy_small_pos,
    with_labels=True,
    node_size=650,
    title='A small copy-model graph (classical m = 1 version)',
)

print(copy_log.head(10).to_string(index=False))


---
## 2. Tunable degree heterogeneity in the classical copy model

For the classical $m=1$ model, the lecture gives
$$
\gamma = \frac{2-p}{1-p}.
$$

We compare the theoretical trend to a simple empirical tail estimate.


In [ ]:
p_values = [0.1, 0.3, 0.5, 0.7, 0.9]
copy_graphs = {p: copy_model(3000, p=p, seed=RANDOM_SEED) for p in p_values}

fig, ax = plt.subplots(figsize=FIGURE_SIZE)
for p, color in zip(p_values, [CATEGORY_PALETTE['blue'], CATEGORY_PALETTE['orange'], CATEGORY_PALETTE['green'], CATEGORY_PALETTE['red'], CATEGORY_PALETTE['purple']]):
    plot_ccdf(
        [degree for _, degree in copy_graphs[p].degree()],
        ax=ax,
        label=f'p = {p}',
        color=color,
        markersize=3,
    )
ax.set_title('Classical copy model: the tail changes with p')
ax.legend(frameon=False)
plt.show()


In [ ]:
gamma_rows = []
for p in p_values:
    degrees = [degree for _, degree in copy_graphs[p].degree()]
    gamma_rows.append({
        'p': p,
        'theoretical gamma': (2 - p) / (1 - p),
        'naive gamma from CCDF tail': gamma_from_ccdf_slope(degrees, kmin=8),
        'max degree': max(degrees),
    })
gamma_df = pd.DataFrame(gamma_rows)
print(gamma_df.round(3).to_string(index=False))


**Interpretation.**
- Small $p$ means more copying through neighbours, so the tail stays broad.
- Large $p$ means more direct attachment to uniformly sampled targets, so heterogeneity weakens.
- The point of the table is the direction of change, not exact parameter fitting.


---
## 3. Copying is not the same as explicit triadic closure

The classical copy model with $m=1$ is too sparse to show much clustering. To compare structure fairly, we use a generalized $m=2$ version and place it next to BA and the random-walk model.


In [ ]:
ba_graph = nx.barabasi_albert_graph(1200, 2, seed=RANDOM_SEED)
copy_graph = generalized_copy_model(1200, 2, p=0.05, seed=RANDOM_SEED)
rw_graph = random_walk_attachment_model(1200, 2, p=0.7, seed=RANDOM_SEED)

compare_df = pd.DataFrame([
    model_summary_row(ba_graph, 'Barabasi-Albert'),
    model_summary_row(copy_graph, 'Generalized copy model (p = 0.05)'),
    model_summary_row(rw_graph, 'Random walk / local search (p = 0.7)'),
])
print(compare_df.round(3).to_string(index=False))

In [ ]:
fig, ax = plt.subplots(figsize=FIGURE_SIZE)
plot_ccdf([degree for _, degree in ba_graph.degree()], ax=ax, label='BA', color=CATEGORY_PALETTE['green'], markersize=3)
plot_ccdf([degree for _, degree in copy_graph.degree()], ax=ax, label='Copy model', color=CATEGORY_PALETTE['blue'], markersize=3)
plot_ccdf([degree for _, degree in rw_graph.degree()], ax=ax, label='Random walk', color=CATEGORY_PALETTE['red'], markersize=3)
ax.set_title('Degree CCDF: BA, copy model, and random-walk model')
ax.legend(frameon=False)
plt.show()


## Takeaway

- The copy model turns **imitation** into a growth rule.
- It can generate broad tails without explicitly measuring degree.
- Its tail depends on $p$, and in the classical formulation the lecture predicts
  $$
  \gamma = \frac{2-p}{1-p}.
  $$
- Copying creates some local structure, but it is not the same mechanism as explicit neighbour-based closure in the random-walk model.
